# Flight Data Platform - Transform (Camada Silver)

Este notebook lê os dados da **camada bronze** (tabelas Delta no catálogo `workspace.default`) e prepara DataFrames para futuras transformações da camada silver.

## Tabelas bronze lidas

| Tabela | Descrição |
| --- | --- |
| `workspace.default.bronze_opensky_states` | Estado atual das aeronaves (OpenSky `/states/all`) |
| `workspace.default.dim_airports` | Dimensão de aeroportos (OurAirports `airports.csv`) |
| `workspace.default.bronze_opensky_departures` | Partidas do SBGR (OpenSky `/flights/departure`) |

## Imports

Importa as funções do Spark SQL (`pyspark.sql.functions`) e o módulo `builtins` do Python. O `builtins` é necessário porque o wildcard import sobrescreve funções nativas como `round`.

In [0]:
# Imports globais: funções do Spark SQL e builtins do Python
# (builtins é necessário pois 'from pyspark.sql.functions import *' sobrescreve o round nativo)
from pyspark.sql.functions import *
import builtins

## 1. Leitura das Tabelas Bronze

Carrega os DataFrames diretamente das tabelas Delta no catálogo `workspace.default` usando `spark.table()`. Cada tabela é lida em uma célula separada para facilitar o debugging individual.

In [0]:
# Ler tabela bronze: estados das aeronaves (OpenSky states/all)
df_states = spark.table("workspace.default.bronze_opensky_states")
print(f"bronze_opensky_states: {df_states.count()} linhas")
display(df_states.limit(10))

In [0]:
# Ler tabela dimensão: aeroportos (OurAirports airports.csv)
df_airports = spark.table("workspace.default.dim_airports")
print(f"dim_airports: {df_airports.count()} linhas")
display(df_airports.limit(10))

In [0]:
# Ler tabela bronze: partidas do SBGR (OpenSky flights/departure)
df_departures = spark.table("workspace.default.bronze_opensky_departures")
print(f"bronze_opensky_departures: {df_departures.count()} linhas")
display(df_departures.limit(10))

## 2. Análise de Nulos

A função `null_report` percorre todas as colunas de cada DataFrame, conta os valores nulos e calcula a porcentagem. O resultado é uma tabela mostrando apenas as colunas que contêm ao menos 1 valor nulo, ordenada por porcentagem decrescente.

In [0]:
def null_report(df, table_name):
    """Conta nulos por coluna e retorna apenas colunas que têm ao menos 1 valor nulo."""
    total = df.count()
    rows = []
    cols = df.columns  # cacheia schema para evitar Analyze RPC no loop
    for c in cols:
        nulls = df.select(count(when(col(c).isNull(), c)).alias("nulls")).collect()[0]["nulls"]
        if nulls > 0:
            pct = (nulls / total) * 100
            rows.append((table_name, c, nulls, builtins.round(pct, 2)))
    return rows

all_rows = []
for df_obj, name in [(df_states, "bronze_opensky_states"), (df_airports, "dim_airports"), (df_departures, "bronze_opensky_departures")]:
    all_rows.extend(null_report(df_obj, name))

if all_rows:
    spark.createDataFrame(all_rows, ["tabela", "coluna", "nulos", "%_nulos"]).orderBy("tabela", "%_nulos", ascending=False).display()
else:
    print("Nenhuma coluna nula encontrada!")

## 3. Limpeza - Remoção de Colunas 100% Nulas

A função `drop_all_null_columns` percorre cada coluna do DataFrame, conta os nulos, e remove aquelas onde **todas as linhas** são nulas. Isso garante que apenas colunas com dados reais permaneçam na camada silver.

In [0]:
def drop_all_null_columns(df, table_name):
    """Detecta e remove colunas onde TODAS as linhas são nulas (100%)."""
    total = df.count()
    cols = df.columns  # cacheia schema para evitar Analyze RPC no loop
    cols_to_drop = []
    for c in cols:
        nulls = df.select(count(when(col(c).isNull(), c)).alias("nulls")).collect()[0]["nulls"]
        if nulls == total:
            cols_to_drop.append(c)
    if cols_to_drop:
        print(f"[{table_name}] Removendo {len(cols_to_drop)} coluna(s) 100% nulas: {cols_to_drop}")
        return df.drop(*cols_to_drop)
    else:
        print(f"[{table_name}] Nenhuma coluna 100% nula.")
        return df

df_states = drop_all_null_columns(df_states, "bronze_opensky_states")
df_airports = drop_all_null_columns(df_airports, "dim_airports")
df_departures = drop_all_null_columns(df_departures, "bronze_opensky_departures")

## 4. Visualização dos DataFrames Limpos

Exibe as primeiras 10 linhas de cada DataFrame após a remoção das colunas 100% nulas.

In [0]:
# Exibe os 3 DataFrames após a remoção de colunas 100% nulas
print("=== bronze_opensky_states ===")
display(df_states.limit(10))
print("=== dim_airports ===")
display(df_airports.limit(10))
print("=== bronze_opensky_departures ===")
display(df_departures.limit(10))